In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
 !pip install albumentations thop medpy scikit-image -q

In [2]:
 !pip install albumentations thop medpy scikit-image -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix, roc_auc_score, matthews_corrcoef
from thop import profile
import matplotlib.pyplot as plt
import time
from medpy.metric.binary import hd95
from skimage.morphology import skeletonize

# ==========================================
# 1. CONFIGURATION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 512
BATCH_SIZE = 4 # Increased for better gradient stability
EPOCHS = 100 
LEARNING_RATE = 1e-4
OUTPUT_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. ARCHITECTURE
# ==========================================

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_l = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_l(x)
        psi = self.relu(g1 + x1)
        return x * self.psi(psi)

class ResidualMSBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 4
        self.branch1 = nn.Conv2d(in_ch, mid, 1)
        self.branch3 = nn.Conv2d(in_ch, mid, 3, padding=1)
        self.branch5 = nn.Conv2d(in_ch, mid, 5, padding=2)
        self.branch7 = nn.Conv2d(in_ch, mid, 7, padding=3)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        res = torch.cat([self.branch1(x), self.branch3(x), self.branch5(x), self.branch7(x)], dim=1)
        return self.relu(self.bn(res) + self.shortcut(x))

class LMFRNet_V2(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ResidualMSBlock(3, 64)
        self.enc2 = ResidualMSBlock(64, 128)
        self.enc3 = ResidualMSBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualMSBlock(256, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.att3 = AttentionGate(256, 256, 128)
        self.dec3 = ResidualMSBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.att2 = AttentionGate(128, 128, 64)
        self.dec2 = ResidualMSBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.att1 = AttentionGate(64, 64, 32)
        self.dec1 = ResidualMSBlock(128, 64)
        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x); p1 = self.pool(e1)
        e2 = self.enc2(p1); p2 = self.pool(e2)
        e3 = self.enc3(p2); p3 = self.pool(e3)
        b = self.bottleneck(p3)
        d3 = self.up3(b); x3 = self.att3(d3, e3); d3 = self.dec3(torch.cat([d3, x3], 1))
        d2 = self.up2(d3); x2 = self.att2(d2, e2); d2 = self.dec2(torch.cat([d2, x2], 1))
        d1 = self.up1(d2); x1 = self.att1(d1, e1); d1 = self.dec1(torch.cat([d1, x1], 1))
        return self.final(d1)

# ==========================================
# 3. ADVANCED METRICS CALCULATION
# ==========================================

def get_cldice(v_p, v_l):
    v_p, v_l = v_p.astype(bool), v_l.astype(bool)
    if np.sum(v_p) == 0 or np.sum(v_l) == 0: return 0.0
    s_p, s_l = skeletonize(v_p), skeletonize(v_l)
    tprec = np.sum(s_p * v_l) / (np.sum(s_p) + 1e-7)
    tsens = np.sum(s_l * v_p) / (np.sum(s_l) + 1e-7)
    return 2 * tprec * tsens / (tprec + tsens + 1e-7)

def calculate_metrics(pred_logits, gt):
    probs = torch.sigmoid(pred_logits).detach().cpu().numpy().squeeze()
    pred_bin = (probs > 0.5).astype(np.uint8)
    gt_np = gt.detach().cpu().numpy().squeeze().astype(np.uint8)
    
    y_true, y_pred, y_prob = gt_np.flatten(), pred_bin.flatten(), probs.flatten()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    return {
        "ACC": (tp + tn) / (tp + tn + fp + fn + 1e-7),
        "Dice": (2 * tp) / (2 * tp + fp + fn + 1e-7),
        "IoU": tp / (tp + fp + fn + 1e-7),
        "Sen": tp / (tp + fn + 1e-7),
        "Spe": tn / (tn + fp + 1e-7),
        "AUC": roc_auc_score(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "clDice": get_cldice(pred_bin, gt_np),
        "HD95": hd95(pred_bin, gt_np) if np.sum(pred_bin) > 0 else 100.0
    }

# ==========================================
# 4. LOSS & DATASET
# ==========================================

class DiceBCEWithLogits(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        bce = self.bce(pred, target)
        pred = torch.sigmoid(pred)
        inter = (pred * target).sum()
        dice = 1 - (2. * inter + 1e-6) / (pred.sum() + target.sum() + 1e-6)
        return bce + dice

class DriveDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=True):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.tf = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.CLAHE(p=1.0), # Critical for seeing thin vessels
            A.HorizontalFlip(p=0.5) if augment else A.NoOp(),
            A.Normalize(), ToTensorV2()
        ])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        
        # Proper filename matching for DRIVE masks
        mask_name = f"{self.images[idx].split('_')[0]}_manual1.gif"
        mask = np.array(Image.open(os.path.join(self.mask_dir, mask_name)))
        
        aug = self.tf(image=img, mask=mask)
        return aug['image'], aug['mask'].unsqueeze(0).float() / 255.0

# ==========================================
# 5. EXECUTION LOOP
# ==========================================

train_img = "/kaggle/input/datasets/abhinav2807/retinal44/DRIVE CHASE STARE DATASET/DRIVE/training/images"
train_mask = "/kaggle/input/datasets/abhinav2807/retinal44/DRIVE CHASE STARE DATASET/DRIVE/training/1st_manual"

loader = DataLoader(DriveDataset(train_img, train_mask), batch_size=BATCH_SIZE, shuffle=True)
model = LMFRNet_V2().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = DiceBCEWithLogits()

# Profile Complexity
dummy = torch.randn(1, 3, 512, 512).to(device)
flops, params = profile(model, inputs=(dummy,), verbose=False)
print(f"Params: {params/1e6:.2f}M | GFLOPs: {flops/1e9:.2f}")

for epoch in range(EPOCHS):
    model.train()
    for img, mask in loader:
        img, mask = img.to(device), mask.to(device)
        optimizer.zero_grad()
        out = model(img)
        loss = criterion(out, mask)
        loss.backward()
        optimizer.step()
    
    # Save a full visual dashboard every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            img, mask = next(iter(loader))
            img_dev = img.to(device)
            out = model(img_dev)
            
            # Select first sample in batch
            orig = img[0].permute(1, 2, 0).cpu().numpy()
            # De-normalize for viewing
            orig = (orig * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
            gt_view = mask[0, 0].cpu().numpy()
            pred_view = torch.sigmoid(out[0, 0]).cpu().numpy()
            
            res = calculate_metrics(out[0:1], mask[0:1])
            
            # Create Comparison Dashboard
            fig, ax = plt.subplots(1, 3, figsize=(18, 6))
            ax[0].imshow(orig); ax[0].set_title("Input Image")
            ax[1].imshow(gt_view, cmap='gray'); ax[1].set_title("Ground Truth (Vessels)")
            ax[2].imshow(pred_view > 0.5, cmap='gray'); ax[2].set_title(f"Prediction (Dice: {res['Dice']:.4f})")
            for a in ax: a.axis('off')
            
            plt.suptitle(f"Epoch {epoch+1} | AUC: {res['AUC']:.4f} | clDice: {res['clDice']:.4f}")
            plt.savefig(f"{OUTPUT_DIR}/epoch_{epoch+1}_dashboard.png")
            plt.close()
            
            print(f"Epoch {epoch+1} Saved. Dice: {res['Dice']:.4f} | HD95: {res['HD95']:.2f}")

print("\n--- FINAL EVALUATION METRICS ---")
final_metrics = calculate_metrics(out, mask)
for k, v in final_metrics.items():
    print(f"{k}: {v:.4f}")

Params: 8.36M | GFLOPs: 207.90
Epoch 10 Saved. Dice: 0.6941 | HD95: 13.45
Epoch 20 Saved. Dice: 0.7505 | HD95: 9.00
Epoch 30 Saved. Dice: 0.7411 | HD95: 4.47
Epoch 40 Saved. Dice: 0.7656 | HD95: 9.85
Epoch 50 Saved. Dice: 0.7654 | HD95: 14.21
Epoch 60 Saved. Dice: 0.7735 | HD95: 12.04
Epoch 70 Saved. Dice: 0.7755 | HD95: 11.05
Epoch 80 Saved. Dice: 0.8414 | HD95: 2.00
Epoch 90 Saved. Dice: 0.8418 | HD95: 5.00
Epoch 100 Saved. Dice: 0.8166 | HD95: 8.49

--- FINAL EVALUATION METRICS ---
ACC: 0.9726
Dice: 0.8351
IoU: 0.7169
Sen: 0.8130
Spe: 0.9875
AUC: 0.9808
MCC: 0.8206
clDice: 0.8124
HD95: 1.0000
